<p align="center"><a href="https://www.plus2net.com/python/pandas-dataframe-select_dtypes.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Pandas select_dtypes(): Select Columns by Type

Run cells in order in Colab. All datasets are synthetic and embedded. Save a copy in Drive to keep edits. Try the exercise before its solution. Dtype inference and output labels may vary by Pandas version; the examples explicitly type the text columns.

## Create a table with several dtypes

Inspect dtypes before choosing selectors. These dates explicitly use day/month/year, so parsing does not rely on locale or ambiguous inference. Text is explicitly stored as object in this first table to demonstrate the object selector.

In [ ]:
import pandas as pd
import numpy as np
def student_data():
    df = pd.DataFrame({
        "NAME": pd.Series(["Ravi", "Raju", "Alex", "Ron", "King", "Jack"], dtype="object"),
        "ID": [1, 2, 3, 4, 5, 6],
        "MATH": [80, 40, 70, 70, 70, 30],
        "Avg_mark": [45.5, 48.09, 50.12, 55.1, 50.6, 55.6],
        "dt_start": pd.to_datetime(["1/1/2020", "2/1/2020", "5/1/2020", "11/7/2020", "15/8/2020", "31/12/2020"], format="%d/%m/%Y"),
        "Exam": [True, False, True, True, False, False],
        "dt": pd.to_timedelta(range(6), unit="D"),
        "grade": pd.Categorical(["a", "c", "b", "b", "b", "c"])
    })
    return df
df = student_data()
print(df.dtypes)
assert df.loc[3, "dt_start"] == pd.Timestamp("2020-07-11")

**Expected output**

```text
NAME                object
ID                   int64
MATH                 int64
Avg_mark           float64
dt_start    datetime64[us]
Exam                  bool
dt          timedelta64[s]
grade             category
dtype: object
```

## Include a specific integer dtype

include="int64" selects the columns with that compatible dtype. It selects columns, not rows, and does not convert numeric text to numbers. Use broader families when exact storage widths are not part of the requirement.

In [ ]:
df = student_data()
result = df.select_dtypes(include="int64")
print(result)
assert result.columns.tolist() == ["ID", "MATH"]

**Expected output**

```text
   ID  MATH
0   1    80
1   2    40
2   3    70
3   4    70
4   5    70
5   6    30
```

## Include several types

Supply a list to select more than one dtype. Boolean columns are not automatically included by the number family; choose bool explicitly when intended.

In [ ]:
df = student_data()
result = df.select_dtypes(include=["int64", "bool"])
print(result)
assert result.columns.tolist() == ["ID", "MATH", "Exam"]

**Expected output**

```text
   ID  MATH   Exam
0   1    80   True
1   2    40  False
2   3    70   True
3   4    70   True
4   5    70  False
5   6    30  False
```

## Exclude integer and float columns

exclude returns the remaining columns. No source data are deleted. Explicit int64/float64 exclusions do not mean every possible numeric storage dtype.

In [ ]:
df = student_data()
result = df.select_dtypes(exclude=["int64", "float64"])
print(result)
assert result.columns.tolist() == ["NAME", "dt_start", "Exam", "dt", "grade"]

**Expected output**

```text
   NAME   dt_start   Exam     dt grade
0  Ravi 2020-01-01   True 0 days     a
1  Raju 2020-01-02  False 1 days     c
2  Alex 2020-01-05   True 2 days     b
3   Ron 2020-07-11   True 3 days     b
4  King 2020-08-15  False 4 days     b
5  Jack 2020-12-31  False 5 days     c
```

## Select numeric families and review their meaning

The number family covers numeric dtypes, and can include timedeltas. Exclude timedelta when selecting ordinary measurements. Numeric identifiers may still be selected; dtype alone cannot distinguish an ID from a score.

In [ ]:
df = student_data()
numeric = df.select_dtypes(include="number", exclude="timedelta")
print(numeric)
assert numeric.columns.tolist() == ["ID", "MATH", "Avg_mark"]
measurements = numeric.drop(columns=["ID"])
print("Measurement means:")
print(measurements.mean())

**Expected output**

```text
   ID  MATH  Avg_mark
0   1    80     45.50
1   2    40     48.09
2   3    70     50.12
3   4    70     55.10
4   5    70     50.60
5   6    30     55.60
Measurement means:
MATH        60.000
Avg_mark    50.835
dtype: float64
```

## Select dates, durations and categories

Use datetime, timedelta and category selectors for their respective columns. Timezone-aware datetime columns can be selected separately with datetimetz. For conversion rules, see <a href="https://www.plus2net.com/python/pandas-dt-to_datetime.php">to_datetime()</a> and <a href="https://www.plus2net.com/python/pandas-dt-to_timedelta.php">to_timedelta()</a>.

In [ ]:
df = student_data()
print("Dates:")
print(df.select_dtypes(include="datetime"))
print("Durations:")
print(df.select_dtypes(include="timedelta"))
print("Categories:")
print(df.select_dtypes(include="category"))
assert df.select_dtypes(include="datetime").columns.tolist() == ["dt_start"]

**Expected output**

```text
Dates:
    dt_start
0 2020-01-01
1 2020-01-02
2 2020-01-05
3 2020-07-11
4 2020-08-15
5 2020-12-31
Durations:
      dt
0 0 days
1 1 days
2 2 days
3 3 days
4 4 days
5 5 days
Categories:
  grade
0     a
1     c
2     b
3     b
4     b
5     c
```

## Inspect object and dedicated string columns separately

object is not a text-only guarantee: it can store mixed numbers, lists or other Python objects. Dedicated string dtypes are separate from object; inference differs across Pandas versions. The example uses explicit object and string dtypes rather than depending on inference.

In [ ]:
table = pd.DataFrame({
    "object_text": pd.Series(["Pen", "Book"], dtype="object"),
    "mixed": pd.Series(["A", 120], dtype="object"),
    "string_text": pd.Series(["Mug", None], dtype="string"),
    "quantity": [2, 3]
})
print(table.dtypes)
print("Object columns:")
print(table.select_dtypes(include="object"))
print("Dedicated string columns:")
print(table.select_dtypes(include="string"))
assert table.select_dtypes(include="object").columns.tolist() == ["object_text", "mixed"]
assert table.select_dtypes(include="string").columns.tolist() == ["string_text"]

**Expected output**

```text
object_text    object
mixed          object
string_text    string
quantity        int64
dtype: object
Object columns:
  object_text mixed
0         Pen     A
1        Book   120
Dedicated string columns:
  string_text
0         Mug
1        <NA>
```

## Select nullable integers and Boolean values

Nullable Int64 and boolean dtypes support missing values. Test the selected column names rather than assuming all numeric columns use NumPy int64. See <a href="https://www.plus2net.com/python/pandas-dataframe-dtypes.php">dtype inspection</a>.

In [ ]:
table = pd.DataFrame({"count": pd.Series([1, None], dtype="Int64"), "price": pd.Series([5.5, None], dtype="Float64"), "active": pd.Series([True, None], dtype="boolean")})
print(table.select_dtypes(include="number"))
print(table.select_dtypes(include="bool"))
assert table.select_dtypes(include="number").columns.tolist() == ["count", "price"]
assert table.select_dtypes(include="bool").columns.tolist() == ["active"]

**Expected output**

```text
   count  price
0      1    5.5
1   <NA>   <NA>
   active
0    True
1    <NA>
```

## Handle missing selectors and overlapping selectors

At least one include or exclude selector is required. Overlapping include/exclude specifications raise an error. A valid selector with no matching columns returns an empty-column DataFrame, preserving the row index.

In [ ]:
df = student_data()
for options in [{}, {"include": ["int64"], "exclude": ["int64"]}]:
    try:
        df.select_dtypes(**options)
    except ValueError as error:
        print(type(error).__name__ + ": " + str(error))
no_match = df.select_dtypes(include="complex")
print("No-match shape:", no_match.shape)
assert no_match.shape == (6, 0)

**Expected output**

```text
ValueError: at least one of include or exclude must be nonempty
ValueError: include and exclude overlap on frozenset({<class 'numpy.int64'>})
No-match shape: (6, 0)
```

## Practical workflow: prepare sales types before analysis

This synthetic extract has text identifiers, mixed numeric input and date text. Define the business schema and convert the specified fields before selecting numeric measurements. Selecting dtypes alone would miss unparsed numbers and include any numeric IDs.

In [ ]:
raw = pd.DataFrame({"order_id": pd.Series(["0012", "0045", "0070"], dtype="string"), "quantity": ["2", "bad", "3"], "price": ["5.50", "8.00", "4.00"], "date": ["2026-01-02", "bad", "2026-01-04"]})
cleaned = raw.copy()
quantity = pd.to_numeric(raw["quantity"], errors="coerce")
integer_quantity = quantity.notna() & quantity.ge(0) & quantity.mod(1).eq(0)
cleaned["quantity"] = quantity.where(integer_quantity).astype("Int64")
cleaned["price"] = pd.to_numeric(raw["price"], errors="coerce").astype("Float64")
cleaned["date"] = pd.to_datetime(raw["date"], format="%Y-%m-%d", errors="coerce")
print(cleaned.dtypes)
numeric = cleaned.select_dtypes(include="number")
print(numeric)
assert numeric.columns.tolist() == ["quantity", "price"]
assert cleaned["order_id"].iloc[0] == "0012"

**Expected output**

```text
order_id            string
quantity             Int64
price              Float64
date        datetime64[us]
dtype: object
   quantity  price
0         2    5.5
1      <NA>    8.0
2         3    4.0
```

## Validate values, then calculate with intended columns

A numeric dtype does not guarantee finite, non-negative or complete values. Validate the required fields and preserve failed rows. Use explicit quantity and price columns for revenue rather than blindly summing all numeric columns.

In [ ]:
finite_price = pd.Series(np.isfinite(cleaned["price"].to_numpy(dtype=float, na_value=np.nan)), index=cleaned.index)
valid = (cleaned[["quantity", "price", "date"]].notna().all(axis=1)
         & finite_price & cleaned["price"].ge(0)).fillna(False)
accepted = cleaned.loc[valid].copy()
accepted["revenue"] = accepted["quantity"] * accepted["price"]
print(accepted)
print("Review rows:")
print(raw.loc[~valid])
assert accepted["order_id"].tolist() == ["0012", "0070"]
assert accepted["revenue"].sum() == 23
# Optional Colab export:
# accepted.to_csv("typed_sales.csv", index=False)

**Expected output**

```text
  order_id  quantity  price       date  revenue
0     0012         2    5.5 2026-01-02     11.0
2     0070         3    4.0 2026-01-04     12.0
Review rows:
  order_id quantity price date
1     0045      bad  8.00  bad
```

## Selection, conversion and validation are separate steps

select_dtypes() chooses columns by stored type. <a href="https://www.plus2net.com/python/pandas-dataframe-astype.php">astype()</a> converts a chosen type; numeric and date parsers handle input formats. Range, format and completeness checks establish usability. Use <a href="https://www.plus2net.com/python/pandas-dataframe-info.php">info()</a> to inspect the table before and after changes. Preserve original input so failed conversions can be reviewed.

## Common questions

<strong>Does select_dtypes() change types?</strong> No. <strong>Why is numeric text missing from the numeric selection?</strong> It has not been parsed into a numeric dtype. <strong>Why did an ID appear?</strong> Numeric storage does not imply measurement meaning. <strong>Why is object not all text?</strong> It allows arbitrary Python objects. <strong>Why did text selection change after a Pandas upgrade?</strong> String inference can change; inspect dtypes and use an explicit schema. <strong>How do I store changes?</strong> Assign converted fields explicitly to the intended DataFrame; do not assume editing a selected table updates its source.

## Exercise: select measurements and a flag

Create code as nullable string, quantity as nullable Int64, price as float and active as bool. Select numeric fields and Boolean fields separately. Expect quantity/price and active respectively.

## Exercise solution

The explicit text identifier remains outside the numeric selection.

In [ ]:
practice = pd.DataFrame({"code": pd.Series(["0012", "0045"], dtype="string"), "quantity": pd.Series([2, None], dtype="Int64"), "price": [5.5, 8.0], "active": [True, False]})
print(practice.select_dtypes(include="number"))
print(practice.select_dtypes(include="bool"))
assert practice.select_dtypes(include="number").columns.tolist() == ["quantity", "price"]
assert practice.select_dtypes(include="bool").columns.tolist() == ["active"]

**Expected output**

```text
   quantity  price
0         2    5.5
1      <NA>    8.0
   active
0    True
1   False
```

## Practice in Google Colab

<a class="btn btn-outline-primary" href="https://colab.research.google.com/github/plus2net/pandas/blob/main/pandas_select_dtypes_practice.ipynb" target="_blank" rel="noopener">Open in Google Colab</a> <a class="btn btn-outline-secondary" href="https://github.com/plus2net/pandas/blob/main/pandas_select_dtypes_practice.ipynb" target="_blank" rel="noopener">View on GitHub</a><br>Run the examples in order, change selectors and input types, then compare selected columns and failed conversions. Save your own copy to keep edits. All sample tables are included in the notebook.